In [1]:
if "moved_up_dir" not in globals():
    # Code that should run once
    %cd ..
    moved_up_dir = True
else:
    print("Skipping — already executed this session.")

/home/woodbkb2/git/nepo-music


In [2]:
import json
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

from apis import spotify as sp
from data_preprocessing import features_calculator as fc
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

load_dotenv()

True

In [3]:

DATA_DIR = Path('./data')
ARTIST_FULL_DATA = DATA_DIR / "artist_full_data"
OUTPUT_DIR = DATA_DIR / "artist_filtered_data" 

EXCLUDE_LIST = {'[unknown]'}
    

In [4]:
import os

In [22]:
count = 0
for file_name in os.listdir(OUTPUT_DIR):
    f = open(OUTPUT_DIR / file_name)
    for l in f:
        count += 1
    print(count)
    
print(f'{count=}')

373
919
1431
1794
2246
2620
3135
3377
3762
4373
4728
5151
5552
5971
6551
6962
7502
7917
8366
8787
9027
9469
9865
10465
10798
11229
11872
12574
13266
13618
14173
14711
15231
15777
16341
16785
17388
17886
18357
19012
19438
19748
20173
20486
20929
21461
21808
22406
22881
23338
23618
24124
24505
24865
25341
25697
26215
26566
26997
27263
27719
28020
28384
28947
29385
29900
30524
30951
31525
32102
32577
32998
33343
33789
34103
34613
35073
35400
36074
36623
37116
37632
38074
38656
39082
39615
39951
40427
40742
41251
41692
42085
42491
42870
43484
43967
44604
45107
45487
45916
46190
46539
47026
47533
47943
48468
48980
49389
49866
50512
50991
51397
51757
51779
52190
52548
52994
53456
53803
54381
54730
55250
55651
56242
56694
57117
57635
58088
58407
58724
59190
59776
60272
60733
61182
61571
62080
62444
62914
63308
63647
64052
64646
65144
65597
66046
66407
66920
67486
67836
68290
68725
69130
69698
70323
70718
71202
71636
72100
72607
73162
73778
74233
74567
74898
75353
75819
76021
76480
76904
77448

In [14]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [28]:
def process_single_file(file_path: Path):
    """Top-level function, importable by worker processes."""
    failures = 0
    processed = 0
    raw_conversions = []

    out_path = OUTPUT_DIR / file_path.name

    try:
        with file_path.open() as f_in, out_path.open('w') as f_out:
            for raw_artist_work_data in f_in:
                try:
                    if raw_artist_work_data == '\n':
                        continue

                    processed += 1
                    artist_work_data = json.loads(raw_artist_work_data)

                    if artist_work_data["artist_name"] in EXCLUDE_LIST:
                        continue

                    if not artist_work_data['works']:
                        continue

                    has_collaborated = False
                    for work in artist_work_data['works']:
                        filtered_collaborators = [
                            collab
                            for collab in work['collaborators']
                            if artist_work_data['mbid'] != collab['mbid']
                        ]
                        if filtered_collaborators:
                            has_collaborated = True
                            break

                    if not has_collaborated:
                        continue

                    calculated_metrics = fc.compute_mb_artist_early_features(
                        artist_work_data, years=5
                    )
                    raw_conversions.append(calculated_metrics)
                    f_out.write(raw_artist_work_data)

                except Exception:
                    failures += 1
    except Exception:
        failures += 1

    return {
        "file": str(file_path),
        "processed": processed,
        "failures": failures,
        "raw_conversions": raw_conversions,
    }

In [29]:
def main():
    files = [f for f in ARTIST_FULL_DATA.iterdir() if f.suffix == ".jsonl"]

    total_processed = 0
    total_failures = 0
    all_raw_conversions = []

    max_workers = None  # or int

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_single_file, f): f for f in files}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Files", unit="file"):
            result = future.result()
            total_processed += result["processed"]
            total_failures += result["failures"]
            all_raw_conversions.extend(result["raw_conversions"])

    print("Finished")
    print("Total processed:", total_processed)
    print("Total failures:", total_failures)


main()

Process ForkServerProcess-73:
Traceback (most recent call last):
  File "/home/woodbkb2/.local/share/uv/python/cpython-3.14.0-linux-x86_64-gnu/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/home/woodbkb2/.local/share/uv/python/cpython-3.14.0-linux-x86_64-gnu/lib/python3.14/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/woodbkb2/.local/share/uv/python/cpython-3.14.0-linux-x86_64-gnu/lib/python3.14/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/home/woodbkb2/.local/share/uv/python/cpython-3.14.0-linux-x86_64-gnu/lib/python3.14/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError: module '__main__' has no attribute 'process_single_file'
Process ForkServerProcess-74:
Traceback (mos

BrokenProcessPool: A child process terminated abruptly, the process pool is not usable anymore

In [30]:
df = pd.DataFrame(raw_conversions)
df.head()

,artist_mbid,artist_name,window_years,debut_date,window_cutoff_date,releases_total,releases_per_year,avg_days_between_releases,release_velocity_releases_per_day,release_velocity_releases_per_year,...,primary_genre,all_genres_str,artist_country,artist_region_city,years_active,debut_year,debut_decade,recency_index,primary_role,all_roles_str
0,5d7df598-bb80-4d73-a6f8-9ed1a4374052,Earl Hines,5,1902-06-01,1907-06-01,2,0.400055,92.000000,0.010870,3.970109,...,None,,United States,Duquesne,123.504449,1902.0,1900.0,0.854117,arranger,"arranger, composer, instrument, performer, voc..."
1,5d7e4d03-5c69-4159-b2b7-092dddbcb09f,Jean‐Michel Defaye,5,1946-01-01,1951-01-01,1,0.200027,NaN,NaN,NaN,...,None,,France,Saint-Mandé,79.917864,1946.0,1940.0,0.568334,arranger,"arranger, composer, conductor, instrument, orc..."
2,5d7f1460-e2a6-4fc0-86df-6f9a8d129416,DIAMANTA,5,2019-04-10,2024-04-10,2,0.399836,0.000000,NaN,NaN,...,None,,Netherlands,Aruba,6.647502,2019.0,2010.0,0.000343,writer,writer
3,5d7f1a01-e08b-44dd-9cab-5a885487250c,Fflur Wyn,5,2010-01-01,2015-01-01,4,0.800110,351.666667,0.002804,1.024234,...,None,,Wales,None,15.917864,2010.0,2010.0,0.000003,vocal,vocal
4,5d7f1f89-4df0-4754-b739-55ab8b9c8b31,Frida Sundemo,5,2001-08-08,2006-08-08,2,0.400055,0.000000,NaN,NaN,...,None,,Sweden,Gothenburg,24.317591,2001.0,2000.0,0.001567,composer,"composer, lyricist, vocal"
